# keypoint-MoSeq noise calibration (interactive)

This notebook runs `keypoint_moseq.calibration.noise_calibration`, which **requires a Jupyter widget UI** (see keypoint-MoSeq docs).

## How to launch

Recommended: use the repo CLI helper to open this notebook with project paths pre-filled:

```bash
python scripts/run_kpms.py --project-path /path/to/external_project --launch-noise-calibration
```

Or launch Jupyter manually and set `project_path` / `workspace_config_path` in the next cell.

Notes:
- You may need `ipympl` for `%matplotlib widget`.
- `video_dir` must point at the external project’s `videos/` folder.


In [ ]:
import os
from pathlib import Path

import keypoint_moseq as kpms

# Allow importing kpms_utils when running Jupyter from repo root
REPO_ROOT = Path.cwd()
try:
    from kpms_utils.path_utils import get_kpms_project_dir, get_pose_data_dir, get_video_dir
    from kpms_utils.config_utils import load_yaml_config
except Exception as exc:  # pragma: no cover
    raise RuntimeError(
        "Could not import kpms_utils. Make sure you launched Jupyter from the repo root."
    ) from exc

# Values can be injected by the CLI launcher (see scripts/run_kpms.py --launch-noise-calibration).
project_path = Path(os.environ.get("KPMS_PROJECT_PATH", "")).expanduser()
workspace_config_path = os.environ.get("KPMS_WORKSPACE_CONFIG", "configs/config_example.yml")
use_filtered = os.environ.get("KPMS_USE_FILTERED", "0") == "1"

# If env vars weren’t provided, set them manually here:
if not str(project_path):
    project_path = Path("/path/to/external_project")

project_path = project_path.resolve()
workspace_config_path = str(Path(workspace_config_path))

kpms_project_dir = get_kpms_project_dir(project_path)
pose_data_dir = get_pose_data_dir(project_path, use_filtered=use_filtered)
video_dir = get_video_dir(project_path)

print("External project :", project_path)
print("KPMS project dir:", kpms_project_dir)
print("Pose data dir   :", pose_data_dir)
print("Video dir       :", video_dir)
print("Workspace config:", workspace_config_path)

In [ ]:
# Load the workspace config (tracker format, extension, recursive search, etc.)
workspace_config = load_yaml_config(workspace_config_path)

fmt = workspace_config.get("pose_estimation_format", "deeplabcut")
ext = workspace_config.get("pose_file_extension") or None
recursive = workspace_config.get("recursive_search", True)

# Ensure KPMS config.yml exists (created by scripts/run_kpms.py prepare step).
# If it doesn’t exist, create it here with minimal required fields.
if not (kpms_project_dir / "config.yml").exists():
    kpms_project_dir.parent.mkdir(parents=True, exist_ok=True)
    kpms.setup_project(
        str(kpms_project_dir),
        overwrite=False,
        bodyparts=workspace_config.get("bodyparts"),
        use_bodyparts=workspace_config.get("use_bodyparts"),
        skeleton=workspace_config.get("skeleton"),
        anterior_bodyparts=workspace_config.get("anterior_bodyparts"),
        posterior_bodyparts=workspace_config.get("posterior_bodyparts"),
        video_dir=str(video_dir),
        fps=workspace_config.get("fps"),
    )

kpms_config = kpms.load_config(str(kpms_project_dir))

# Load keypoints
coordinates, confidences, bodyparts = kpms.load_keypoints(
    str(pose_data_dir),
    format=fmt,
    extension=ext,
    recursive=recursive,
)

print("Loaded recordings:", len(coordinates))
print("Bodyparts (loader):", len(bodyparts))
print("Bodyparts (config):", len(kpms_config.get('bodyparts', [])))

In [ ]:
# Calibration UI (Jupyter widget)

# If you see an error here, install ipympl and restart the kernel:
#   pip install ipympl

%matplotlib widget
kpms.noise_calibration(str(kpms_project_dir), coordinates, confidences, **kpms_config)